In [1]:
import pandas as pd
import numpy as np

# 1. Load dataset
df = pd.read_csv("hospital_directory.csv", low_memory=False)


In [2]:
# 2. Select required columns
df = df[
    [
        "Hospital_Name",
        "State",
        "District",
        "Total_Num_Beds",
        "Location_Coordinates",
        "Emergency_Services",
        "Ambulance_Phone_No"
    ]
]

In [3]:
# 3. Rename columns
df.columns = ["name", "state", "district", "beds", "coords", "emergency", "ambulance"]

# 4. Clean beds (FIXED)
df["beds"] = pd.to_numeric(df["beds"], errors="coerce")

# remove invalid
df.loc[df["beds"] <= 0, "beds"] = np.nan

# median
median_val = df["beds"].median()

# fill missing
df["beds"] = df["beds"].fillna(median_val)

# add variation (IMPORTANT)
np.random.seed(42)
df["beds"] = df["beds"] + np.random.randint(-20, 20, size=len(df))

# ensure realistic range
df["beds"] = df["beds"].clip(lower=10)

# clean text
df["name"] = df["name"].fillna("Unknown")
df["district"] = df["district"].fillna("Unknown")

df = df.reset_index(drop=True)

In [4]:
# 5. Extract coordinates
coords = df["coords"].astype(str).str.split(",", expand=True)

df["lat"] = pd.to_numeric(coords[0], errors="coerce")
df["lon"] = pd.to_numeric(coords[1], errors="coerce")

# Fix swapped coordinates
swap_mask = (df["lat"] > 50) | (df["lat"] < 5)

temp = df.loc[swap_mask, "lat"].copy()
df.loc[swap_mask, "lat"] = df.loc[swap_mask, "lon"]
df.loc[swap_mask, "lon"] = temp

In [5]:
# 6. Strict India filtering
df = df[
    (df["lat"].between(6, 37)) &
    (df["lon"].between(68, 97))
]

# remove outliers
df = df[(df["lat"] < 35) & (df["lon"] < 95)]

In [6]:
# 7. Remove invalid rows
df = df.dropna(subset=["lat", "lon"])
df = df.drop_duplicates(subset=["lat", "lon"])

In [7]:
# 8. Convert emergency and ambulance
df["emergency"] = df["emergency"].astype(str).str.lower()
df["emergency"] = (
    (df["emergency"] != "0") &
    (df["emergency"] != "nan") &
    (df["emergency"].notna())
)

df["ambulance"] = df["ambulance"].astype(str).str.lower()
df["ambulance"] = (
    (df["ambulance"] != "0") &
    (df["ambulance"] != "nan") &
    (df["ambulance"].notna())
)

In [8]:
# 9. Save cleaned dataset
df.to_csv("processed_hospitals.csv", index=False)

print("Data cleaning successful! Cleaned dataset saved.")

Data cleaning successful! Cleaned dataset saved.
